In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PhishGuard AI — RQ5: Deep Learning Models (BiLSTM + TextCNN)
# Input : dl_tokenizer.pkl + train/val/test.csv (text_cleaned_transformer)
# Output: .pt checkpoints | training_history.csv | metrics | diagrams
# ═══════════════════════════════════════════════════════════════════════════════

# ═══ CELL 1 — Reproducibility Block ═══
import os, random, numpy as np, torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Reproducibility block applied (SEED=42) | Device: {DEVICE}")
if DEVICE.type == "cpu":
    print("⚠ GPU not detected — enable GPU for faster training (Runtime → T4 GPU)")


# ═══ CELL 2 — Drive mount + folder tree ═══
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR    = "/content/drive/MyDrive/NLP FINAL"
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
STAGE_DIR   = os.path.join(BASE_DIR, "rq5_deep_learning")
RQ3_MODELS  = os.path.join(BASE_DIR, "rq3_feature_engineering", "models")

MODELS_DIR   = os.path.join(STAGE_DIR, "models")
LOGS_DIR     = os.path.join(STAGE_DIR, "logs")
RESULTS_DIR  = os.path.join(STAGE_DIR, "results")
DIAGRAMS_DIR = os.path.join(STAGE_DIR, "diagrams")

for folder in [MODELS_DIR, LOGS_DIR, RESULTS_DIR, DIAGRAMS_DIR]:
    os.makedirs(folder, exist_ok=True)

TRAIN_PATH = os.path.join(DATASET_DIR, "train.csv")
VAL_PATH   = os.path.join(DATASET_DIR, "val.csv")
TEST_PATH  = os.path.join(DATASET_DIR, "test.csv")
DLTOK_PATH = os.path.join(RQ3_MODELS, "dl_tokenizer.pkl")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, DLTOK_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing: {p}\nRun Stage 0 & RQ3 first.")

print(f"✓ STAGE_DIR: {STAGE_DIR}")


# ═══ CELL 3 — Install + imports ═══
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "joblib", "matplotlib", "seaborn", "scikit-learn", "tensorflow"])

import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import platform
from datetime import datetime
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve,
)

DPI = 300
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams["figure.dpi"] = DPI
plt.rcParams["savefig.dpi"] = DPI


# ═══ CELL 4 — Hyperparameters ═══
NUM_EPOCHS   = 5          # plan: 3–5 epochs
BATCH_SIZE   = 64
EMBED_DIM    = 128
HIDDEN_DIM   = 128
NUM_FILTERS  = 100
FILTER_SIZES = [3, 4, 5]
DROPOUT      = 0.3
LR           = 1e-3
LSTM_LAYERS  = 1


# ═══ CELL 5 — Load data + tokenizer (NEVER refit tokenizer) ═══
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

dl_tokenizer = joblib.load(DLTOK_PATH)   # load only — never .fit_on_texts()

train_texts = train_df["text_cleaned_transformer"].astype(str).tolist()
val_texts   = val_df["text_cleaned_transformer"].astype(str).tolist()
test_texts  = test_df["text_cleaned_transformer"].astype(str).tolist()

y_train = train_df["label"].values.astype(np.float32)
y_val   = val_df["label"].values.astype(np.float32)
y_test  = test_df["label"].values.astype(np.float32)

VOCAB_SIZE = len(dl_tokenizer.word_index) + 1   # +1 for padding index 0

# Pad length = 95th percentile of train sequence lengths (from RQ3 logic)
train_seq_lens = [len(s) for s in dl_tokenizer.texts_to_sequences(train_texts)]
MAX_LEN = int(min(np.percentile(train_seq_lens, 95), 512))
MAX_LEN = max(MAX_LEN, 50)

# Imbalance weight for BCEWithLogitsLoss
n_safe     = int((y_train == 0).sum())
n_phishing = int((y_train == 1).sum())
POS_WEIGHT = n_safe / n_phishing

print(f"Vocab size : {VOCAB_SIZE:,}")
print(f"Max length : {MAX_LEN}  (p95 capped at 512)")
print(f"POS_WEIGHT : {POS_WEIGHT:.4f}")
print(f"Train/Val/Test : {len(y_train):,} / {len(y_val):,} / {len(y_test):,}")


def texts_to_padded(texts):
    seqs = dl_tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")


X_train = texts_to_padded(train_texts)
X_val   = texts_to_padded(val_texts)
X_test  = texts_to_padded(test_texts)

print(f"Padded shape (train): {X_train.shape}")


# ═══ CELL 6 — PyTorch DataLoaders ═══
def make_loader(X, y, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.long)
    y_t = torch.tensor(y, dtype=torch.float32)
    ds  = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(SEED))

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   shuffle=False)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)


# ═══ CELL 7 — Model definitions ═══
class BiLSTMClassifier(nn.Module):
    """Trainable embeddings + bidirectional LSTM."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)                          # (B, L, E)
        _, (h_n, _) = self.lstm(embedded)                     # h_n: (2*layers, B, H)
        hidden = torch.cat((h_n[-2], h_n[-1]), dim=1)        # (B, 2H)
        return self.fc(self.dropout(hidden)).squeeze(-1)      # (B,)


class TextCNNClassifier(nn.Module):
    """Trainable embeddings + multi-kernel 1-D CNN (Kim 2014 style)."""
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes, max_len, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), 1)
        self.max_len = max_len

    def forward(self, x):
        embedded = self.embedding(x).permute(0, 2, 1)         # (B, E, L)
        conv_outs = []
        for conv in self.convs:
            c = torch.relu(conv(embedded)).max(dim=2).values  # (B, F)
            conv_outs.append(c)
        cat = torch.cat(conv_outs, dim=1)                       # (B, F*kernels)
        return self.fc(self.dropout(cat)).squeeze(-1)


print("✓ BiLSTM & TextCNN model classes defined")


# ═══ CELL 8 — Training / evaluation helpers ═══
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []

    with torch.set_grad_enabled(is_train):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(y_batch)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y_batch.cpu().numpy())

    n = len(all_labels)
    avg_loss = total_loss / n
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1, np.array(all_labels), np.array(all_preds), np.array(all_probs)


def compute_test_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy":        round(accuracy_score(y_true, y_pred), 4),
        "precision":       round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":          round(recall_score(y_true, y_pred, zero_division=0), 4),
        "macro_f1":        round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "roc_auc":         round(roc_auc_score(y_true, y_prob), 4),
        "pr_auc":          round(average_precision_score(y_true, y_prob), 4),
        "phishing_recall": round(recall_score(y_true, y_pred, pos_label=1, zero_division=0), 4),
    }


def save_cm_fig(cm, title, filepath, normalize=False):
    cm_plot = cm.astype(float) / cm.sum(axis=1, keepdims=True) if normalize else cm
    fmt = ".2f" if normalize else "d"
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(cm_plot, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=["Legitimate", "Phishing"],
                yticklabels=["Legitimate", "Phishing"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(title + (" (Normalized)" if normalize else " (Raw Counts)"), fontweight="bold")
    plt.tight_layout()
    plt.savefig(filepath, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close()


def save_training_curves(history_df, model_key, display_name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(history_df["epoch"], history_df["train_loss"], "o-", label="Train Loss", color="#3498db")
    axes[0].plot(history_df["epoch"], history_df["val_loss"],   "s-", label="Val Loss",   color="#e74c3c")
    axes[0].set_title(f"{display_name} — Loss", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["train_accuracy"], "o-", label="Train Acc", color="#2ecc71")
    axes[1].plot(history_df["epoch"], history_df["val_accuracy"],   "s-", label="Val Acc",   color="#9b59b6")
    axes[1].plot(history_df["epoch"], history_df["train_f1"],       "o--", label="Train F1", color="#1abc9c", alpha=0.8)
    axes[1].plot(history_df["epoch"], history_df["val_f1"],         "s--", label="Val F1",   color="#f39c12", alpha=0.8)
    axes[1].set_title(f"{display_name} — Accuracy & Macro-F1", fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score"); axes[1].legend()

    fig.suptitle(f"RQ5 Training Curves — {display_name}", fontweight="bold", y=1.02)
    plt.tight_layout()
    path = os.path.join(DIAGRAMS_DIR, f"rq5_{model_key}_training_curves.png")
    plt.savefig(path, dpi=DPI, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  ✓ Saved training curves → rq5_{model_key}_training_curves.png")


# ═══ CELL 9 — Train both models ═══
MODEL_CONFIGS = {
    "bilstm": {
        "display_name": "BiLSTM",
        "family": "Recurrent (Deep Learning)",
        "build_fn": lambda: BiLSTMClassifier(
            VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, num_layers=LSTM_LAYERS, dropout=DROPOUT
        ),
    },
    "textcnn": {
        "display_name": "TextCNN",
        "family": "Convolutional (Deep Learning)",
        "build_fn": lambda: TextCNNClassifier(
            VOCAB_SIZE, EMBED_DIM, NUM_FILTERS, FILTER_SIZES, MAX_LEN, dropout=DROPOUT
        ),
    },
}

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([POS_WEIGHT], dtype=torch.float32).to(DEVICE)
)

all_history_rows    = []
overall_metrics_rows = []
classification_rows  = []
confusion_rows       = []
test_pred_df         = pd.DataFrame({"y_true": y_test.astype(int)})

print(f"\nTraining {len(MODEL_CONFIGS)} DL models for {NUM_EPOCHS} epochs...\n")

for model_key, cfg in MODEL_CONFIGS.items():
    display_name = cfg["display_name"]
    print(f"{'='*60}")
    print(f"  {display_name}")
    print(f"{'='*60}")

    model     = cfg["build_fn"]().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history_rows = []
    best_val_f1  = -1.0
    best_state   = None

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1, _, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc, va_f1, _, _, _ = run_epoch(model, val_loader,   criterion)

        history_rows.append({
            "model_key": model_key, "model_name": display_name,
            "epoch": epoch,
            "train_loss": round(tr_loss, 4), "val_loss": round(va_loss, 4),
            "train_accuracy": round(tr_acc, 4), "val_accuracy": round(va_acc, 4),
            "train_f1": round(tr_f1, 4), "val_f1": round(va_f1, 4),
        })
        print(f"  Epoch {epoch}/{NUM_EPOCHS} | "
              f"train loss={tr_loss:.4f} acc={tr_acc:.4f} f1={tr_f1:.4f} | "
              f"val loss={va_loss:.4f} acc={va_acc:.4f} f1={va_f1:.4f}")

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restore best checkpoint (by val Macro-F1)
    model.load_state_dict(best_state)

    # Save checkpoint (.pt)
    ckpt_path = os.path.join(MODELS_DIR, f"{model_key}.pt")
    torch.save({
        "model_state_dict": model.state_dict(),
        "model_key":        model_key,
        "model_name":       display_name,
        "vocab_size":       VOCAB_SIZE,
        "max_len":          MAX_LEN,
        "embed_dim":        EMBED_DIM,
        "best_val_f1":      round(best_val_f1, 4),
        "hyperparameters": {
            "num_epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
            "lr": LR, "dropout": DROPOUT, "pos_weight": POS_WEIGHT,
        },
    }, ckpt_path)
    print(f"  ✓ Checkpoint saved → {model_key}.pt  (best val F1={best_val_f1:.4f})")

    # Training curves diagram
    history_df = pd.DataFrame(history_rows)
    save_training_curves(history_df, model_key, display_name)
    all_history_rows.extend(history_rows)

    # ── Evaluate on TEST ──
    _, _, _, y_true, y_pred, y_prob = run_epoch(model, test_loader, criterion)
    metrics = compute_test_metrics(y_true, y_pred, y_prob)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print(f"  TEST → Acc={metrics['accuracy']} Macro-F1={metrics['macro_f1']} "
          f"ROC-AUC={metrics['roc_auc']} Phishing-Recall={metrics['phishing_recall']}\n")

    overall_metrics_rows.append({
        "model_key": model_key, "model_name": display_name,
        "model_family": cfg["family"], "eval_set": "test",
        "best_val_f1": round(best_val_f1, 4), **metrics,
    })

    report = classification_report(
        y_true, y_pred, target_names=["Legitimate", "Phishing"],
        output_dict=True, zero_division=0,
    )
    for lbl in ["Legitimate", "Phishing", "macro avg"]:
        key = lbl if lbl != "macro avg" else "macro avg"
        classification_rows.append({
            "model_key": model_key, "model_name": display_name, "class": lbl,
            "precision": round(report[key]["precision"], 4),
            "recall":    round(report[key]["recall"], 4),
            "f1_score":  round(report[key]["f1-score"], 4),
            "support":   int(report[key]["support"]),
        })

    confusion_rows.append({
        "model_key": model_key, "model_name": display_name,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "phishing_recall": metrics["phishing_recall"],
    })

    test_pred_df[f"{model_key}_y_pred"]        = y_pred
    test_pred_df[f"{model_key}_phishing_prob"] = np.round(y_prob, 6)

    save_cm_fig(cm, f"Confusion Matrix — {display_name}",
                os.path.join(DIAGRAMS_DIR, f"rq5_cm_{model_key}_raw.png"), normalize=False)
    save_cm_fig(cm, f"Confusion Matrix — {display_name}",
                os.path.join(DIAGRAMS_DIR, f"rq5_cm_{model_key}_normalized.png"), normalize=True)


# ═══ CELL 10 — Save logs + results CSVs ═══
history_all = pd.DataFrame(all_history_rows)
history_all.to_csv(os.path.join(LOGS_DIR, "training_history.csv"), index=False)

overall_metrics_df = pd.DataFrame(overall_metrics_rows)
overall_metrics_df.to_csv(os.path.join(RESULTS_DIR, "overall_metrics.csv"), index=False)

pd.DataFrame(classification_rows).to_csv(
    os.path.join(RESULTS_DIR, "classification_report.csv"), index=False)
pd.DataFrame(confusion_rows).to_csv(
    os.path.join(RESULTS_DIR, "confusion_matrix.csv"), index=False)
test_pred_df.to_csv(os.path.join(RESULTS_DIR, "test_predictions.csv"), index=False)

paper_table = overall_metrics_df[[
    "model_name", "model_family", "accuracy", "precision", "recall",
    "macro_f1", "roc_auc", "pr_auc", "phishing_recall", "best_val_f1"
]].copy()
paper_table.columns = [
    "Model", "Family", "Accuracy", "Precision", "Recall",
    "Macro-F1", "ROC-AUC", "PR-AUC", "Phishing Recall", "Best Val F1"
]
paper_table.to_csv(os.path.join(RESULTS_DIR, "paper_table_rq5_deep_learning.csv"), index=False)

config = {
    "project":              "PhishGuard AI",
    "stage":                "RQ5 — Deep Learning (BiLSTM + TextCNN)",
    "timestamp_utc":        datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    "seed":                 SEED,
    "device":               str(DEVICE),
    "text_column":          "text_cleaned_transformer",
    "tokenizer":            DLTOK_PATH,
    "tokenizer_refit":      False,
    "vocab_size":           VOCAB_SIZE,
    "max_seq_length":       MAX_LEN,
    "num_epochs":           NUM_EPOCHS,
    "batch_size":           BATCH_SIZE,
    "embed_dim":            EMBED_DIM,
    "hidden_dim":           HIDDEN_DIM,
    "filter_sizes":         str(FILTER_SIZES),
    "num_filters":          NUM_FILTERS,
    "dropout":              DROPOUT,
    "learning_rate":        LR,
    "pos_weight_bce":       round(POS_WEIGHT, 6),
    "checkpoint_format":    "PyTorch .pt",
    "best_checkpoint_rule": "highest val Macro-F1",
    "python_version":       platform.python_version(),
    "torch_version":        torch.__version__,
}
pd.DataFrame(list(config.items()), columns=["parameter", "value"]).to_csv(
    os.path.join(RESULTS_DIR, "experiment_config.csv"), index=False)

print("✓ Saved logs + results CSVs")


# ═══ CELL 11 — Final summary ═══
print("\n" + "=" * 70)
print("RQ5 — DEEP LEARNING COMPLETE ✓")
print("=" * 70)
print(f"\nCheckpoints → {MODELS_DIR}/")
print("  bilstm.pt | textcnn.pt")
print(f"\nLogs → {LOGS_DIR}/training_history.csv")
print(f"\nDiagrams → {DIAGRAMS_DIR}/")
print("  rq5_bilstm_training_curves.png | rq5_textcnn_training_curves.png")
print("  rq5_cm_*_raw.png | rq5_cm_*_normalized.png")
print("\n── Test Set Performance ──")
display(paper_table)
print("\nNext step → RQ6 (DistilBERT fine-tuning)")
print("=" * 70)

✓ Reproducibility block applied (SEED=42) | Device: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ STAGE_DIR: /content/drive/MyDrive/NLP FINAL/rq5_deep_learning
Vocab size : 203,971
Max length : 512  (p95 capped at 512)
POS_WEIGHT : 1.5491
Train/Val/Test : 13,041 / 2,795 / 2,795
Padded shape (train): (13041, 512)
✓ BiLSTM & TextCNN model classes defined

Training 2 DL models for 5 epochs...

  BiLSTM
  Epoch 1/5 | train loss=0.4928 acc=0.8091 f1=0.8031 | val loss=0.3074 acc=0.8984 f1=0.8947
  Epoch 2/5 | train loss=0.2274 acc=0.9275 f1=0.9249 | val loss=0.2385 acc=0.9281 f1=0.9252
  Epoch 3/5 | train loss=0.3063 acc=0.9014 f1=0.8990 | val loss=0.2488 acc=0.9148 f1=0.9121
  Epoch 4/5 | train loss=0.1212 acc=0.9623 f1=0.9608 | val loss=0.2130 acc=0.9388 f1=0.9360
  Epoch 5/5 | train loss=0.0979 acc=0.9677 f1=0.9665 | val loss=0.2363 acc=0.9356 f1=0.9327
  ✓ Checkpoint saved → bilstm.pt  (best val F1=

/tmp/ipykernel_1069/171089972.py:439: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc":        datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),


,Model,Family,Accuracy,Precision,Recall,Macro-F1,ROC-AUC,PR-AUC,Phishing Recall,Best Val F1
0,BiLSTM,Recurrent (Deep Learning),0.9374,0.9160,0.9252,0.9344,0.9831,0.9675,0.9252,0.9360
1,TextCNN,Convolutional (Deep Learning),0.9682,0.9468,0.9735,0.9668,0.9940,0.9882,0.9735,0.9672



Next step → RQ6 (DistilBERT fine-tuning)
